In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    coalesce,
    lpad,
    lit,
    make_date,
    regexp_replace,
    trim
)
from pyspark.sql.types import DoubleType, IntegerType

CATALOG_NAME = "isp"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
VOLUME_NAME = "isp_volumes"
TABLE_NAME = "timeseries_monthly_since_2003"

bronze_table = f"{CATALOG_NAME}.{BRONZE_SCHEMA_NAME}.{TABLE_NAME}"
silver_table = f"{CATALOG_NAME}.{SILVER_SCHEMA_NAME}.{TABLE_NAME}"
checkpoint_path = f"/Volumes/{CATALOG_NAME}/{SILVER_SCHEMA_NAME}/{VOLUME_NAME}/_checkpoints/{TABLE_NAME}"

In [0]:
df_bronze_stream = (
    spark.readStream
    .format("delta")
    .table(bronze_table)
)

In [0]:
id_columns = ["ano", "mes", "mes_ano", "reference_date", "fase"]

metadata_columns = {"ano", "mes", "mes_ano", "fase"}
metric_columns = [
    c for c in df_bronze_stream.columns 
    if c not in metadata_columns and not c.startswith("_")
]

df_cleaned = (
    df_bronze_stream
    .filter(col("ano").isNotNull() & col("mes").between(1, 12))
    .withColumn("ano", col("ano").cast(IntegerType()))
    .withColumn("mes", col("mes").cast(IntegerType()))
    .withColumn("mes_ano", trim(col("mes_ano")))
    .withColumn("reference_date", make_date(col("ano"), col("mes"), lit(1)))
    .withColumn("fase", col("fase").cast(IntegerType()))
    .select(
        col("ano"),
        col("mes"),
        col("mes_ano"),
        col("reference_date"),
        col("fase"),
        *[
            coalesce(
                regexp_replace(col(col_name), ",", ".").cast(DoubleType()),
                lit(0.0)
            ).alias(col_name)
            for col_name in metric_columns
        ]
    )
)

df_silver_transformed = (
    df_cleaned
    .unpivot(
        ids=id_columns,
        values=metric_columns,
        variableColumnName="ocorrencia",
        valueColumnName="taxa"
    )
    .withColumn("_silver_processed_at", current_timestamp())
)

In [0]:
query = (
    df_silver_transformed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

query.awaitTermination()

In [0]:
%sql

SELECT * FROM isp.silver.timeseries_monthly_since_2003